In [ ]:
import os
import sys
import asyncio
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# - - 配置  - -
# 确保您的 API 密钥环境变量已设置（例如 OPENAI_API_KEY）
try:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    print(f"Language model initialized: {llm.model_name}")
except Exception as e:
    print(f"Error initializing language model: {e}", file=sys.stderr)
    print("Please ensure your OPENAI_API_KEY is set correctly.", file=sys.stderr)
    sys.exit(1) # 如果LLM无法初始化则退出


# --- 定义链组件 ---

# 1. 初始生成：创建产品描述的初稿。
# 该链的输入将是一个字典，因此我们更新提示模板。
generation_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Write a short, simple product description for a new smart coffee mug."),
        ("user", "{product_details}")
    ])
    | llm
    | StrOutputParser()
)

# 2. 批评：评估生成的描述并提供反馈。
critique_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """Critique the following product description based on clarity, conciseness, and appeal.
        Provide specific suggestions for improvement."""),
        # 这将从上一步接收“initial_description”。
        ("user", "Product Description to Critique:\n{initial_description}")
    ])
    | llm
    | StrOutputParser()
)

# 3. 细化：根据原来的细节和批评重写描述。
refinement_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """Based on the original product details and the following critique,
        rewrite the product description to be more effective.

        Original Product Details: {product_details}
        Critique: {critique}

        Refined Product Description:"""),
        ("user", "") # 用户输入为空，因为系统消息中提供了上下文
    ])
    | llm
    | StrOutputParser()
)


# --- 构建完整的反射链（重构） ---
# 该链的结构更具可读性和线性性。
full_reflection_chain = (
    RunnablePassthrough.assign(
        initial_description=generation_chain
    )
    | RunnablePassthrough.assign(
        critique=critique_chain
    )
    | refinement_chain
)


# --- 运行链条 ---
async def run_reflection_example(product_details: str):
    """Runs the LangChain reflection example with product details."""
    print(f"\n--- Running Reflection Example for Product: '{product_details}' ---")
    try:
        # 该链现在从一开始就需要一个字典作为输入。
        final_refined_description = await full_reflection_chain.ainvoke(
            {"product_details": product_details}
        )
        print("\n--- Final Refined Product Description ---")
        print(final_refined_description)
    except Exception as e:
        print(f"\nAn error occurred during chain execution: {e}")

if __name__ == "__main__":
    test_product_details = "A mug that keeps coffee hot and can be controlled by a smartphone app."
    asyncio.run(run_reflection_example(test_product_details))